In [45]:
import pandas as pd

df_gmv = pd.read_csv("./test/gmv_clean_sales_infor.csv", encoding='utf-8-sig')
df_leads = pd.read_csv("./test/leads_clean_with_note_phone.csv", encoding='utf-8-sig')
df_unmatched = pd.read_csv("./test/gmv_unmatched.csv", encoding='utf-8-sig', 
                           dtype={"Phone_clean": str, "Phone_extracted_from_note": str})

In [46]:
df_unmatched.columns

Index(['bank day', 'bank time', 'Gateway', 'User Name', 'Phone', 'UID',
       'Package', 'Fixed/ non-fixed', 'Pay Time', 'Real Pay(VND)', 'GMV (RMB)',
       'Payment Method', 'Type', 'Sales', 'Note', 'Month of payment',
       'Unnamed: 16', 'TEAM', 'Full Price(VND)', '采购价格 \n（套餐配置）', '渠道号',
       '首次申请试听日期', '首次申请试听渠道号', '首次购课时间', 'Source', '已批工单', '总 B (被推荐） 课数',
       'A (推荐人） 课数 (送PH)', 'A', 'A (推荐人）课数 (送UK)', 'B', '实际卖课单价 VND',
       '实际卖课单价 RMB', 'PH/UK', '采购价格（包括转介绍赠送课)', 'Nguồn', 'UID_clean',
       'Phone_clean', 'Sales_Abbr', 'Sales_infor', 'Đầu Ngày', 'Họ và tên',
       'Người trực', 'Số điện thoại', 'Tuổi', 'Quốc Gia', 'Note_lead', 'TVTS',
       'Ad_id', 'Nguồn_lead', 'Quốc gia \nQuảng cáo', 'Mẫu Quảng cáo',
       'Mã CRM', 'Sale sau phân', 'Link FB', 'UID_lead', 'TT', 'Trạng thái',
       'Ngày lên học thử', 'Current Binded Sale',
       'Sales in latest system-weighted allocation',
       'Latest system-weighted Allocation Time', 'Allocate reason',
       'Tên sal

In [51]:
df_unmatched["Phone_clean"]

0       84908224672
1       84909517679
2       84389970625
3       84938222653
4       84844534222
           ...     
174     84368776525
175    817035351368
176     84969663003
177     84943111987
178     84943111987
Name: Phone_clean, Length: 179, dtype: object

In [66]:
df_leads = pd.read_csv(
    "./test/leads_clean_with_note_phone.csv",
    dtype={"Phone_clean": str, "Phone_extracted_from_note": str}
)

In [80]:
df_gmv = pd.read_csv("./test/gmv_clean_sales_infor.csv")

In [73]:
df_leads["Phone_clean"] = (
    df_leads["Phone_clean"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
)

In [74]:
df_leads["Phone_clean"]

0               <NA>
1        84919249237
2        84965468525
3        84985951781
4        84972856556
            ...     
16331    84909693803
16332    84986806948
16333    84981971660
16334    84389479460
16335    84935030790
Name: Phone_clean, Length: 16336, dtype: string

In [75]:
# import pandas as pd
import re
from difflib import get_close_matches

# =====================================================
# STEP 1. Normalize phone
# =====================================================

def norm_phone(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = re.sub(r"\D", "", x)  # keep digits only
    return x


df_unmatched["Phone_clean"] = df_unmatched["Phone_clean"].apply(norm_phone)
df_leads["Phone_clean"] = df_leads["Phone_clean"].apply(norm_phone)

# remove empty
unmatched_phones = df_unmatched["Phone_clean"].dropna().unique().tolist()
lead_phones = df_leads["Phone_clean"].dropna().unique().tolist()

# =====================================================
# STEP 2. Fuzzy match using difflib
# =====================================================

matches = []

for p in unmatched_phones:

    if p == "":
        continue

    # lấy 5 match gần nhất
    close = get_close_matches(p, lead_phones, n=5, cutoff=0.80)

    for c in close:
        matches.append({
            "unmatched_phone": p,
            "lead_phone": c
        })

# =====================================================
# STEP 3. Convert to DataFrame
# =====================================================

df_fuzzy = pd.DataFrame(matches)

print("Total fuzzy matches:", len(df_fuzzy))
df_fuzzy.head(30)

Total fuzzy matches: 498


,unmatched_phone,lead_phone
0,84908224672,84983224462
1,84908224672,84982467725
2,84908224672,84977082467
3,84908224672,84968922467
4,84908224672,84902264612
5,84909517679,84909572693
6,84909517679,84909526379
7,84909517679,84908501679
8,84909517679,84904976789
9,84909517679,84904517691


In [79]:
import editdistance

matches = []

for p1 in unmatched_phones:
    if p1 == "":
        continue

    for p2 in lead_phones:

        if abs(len(p1) - len(p2)) <= 2:

            dist = editdistance.eval(p1, p2)

            if dist <= 2:
                matches.append({
                    "unmatched_phone": p1,
                    "lead_phone": p2,
                    "distance": dist
                })

df_fuzzy = pd.DataFrame(matches)
df_fuzzy.sort_values("distance").head(30)

,unmatched_phone,lead_phone,distance
20,84964166015,84964166016,1
12,84966645999,84966645995,1
8,84909610282,84909110282,1
22,84373406103,84334061503,2
23,84944022155,84904022135,2
24,84975567075,84975557095,2
25,84965272428,84965282421,2
26,84984909000,84989809000,2
27,84984909000,84984929006,2
0,84844534222,84844936222,2


In [77]:
# =====================================================
# STEP 1. Clean UID
# =====================================================

def norm_uid(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

df_unmatched["UID_clean"] = df_unmatched["UID_clean"].apply(norm_uid)
df_leads["UID_clean"] = df_leads["UID_clean"].apply(norm_uid)

unmatched_uids = df_unmatched["UID_clean"].dropna().unique().tolist()
lead_uids = df_leads["UID_clean"].dropna().unique().tolist()

# =====================================================
# STEP 2. Fuzzy match (edit distance)
# =====================================================

matches = []

for u1 in unmatched_uids:

    if u1 == "":
        continue

    for u2 in lead_uids:

        # lọc nhanh để giảm cost
        if abs(len(u1) - len(u2)) <= 2:

            dist = editdistance.eval(u1, u2)

            if dist <= 1:
                matches.append({
                    "unmatched_uid": u1,
                    "lead_uid": u2,
                    "distance": dist
                })

# =====================================================
# STEP 3. Result
# =====================================================

df_uid_fuzzy = pd.DataFrame(matches)
df_uid_fuzzy = df_uid_fuzzy.sort_values("distance")

print("Total fuzzy UID matches:", len(df_uid_fuzzy))
df_uid_fuzzy.head(30)

Total fuzzy UID matches: 17


,unmatched_uid,lead_uid,distance
0,3311919278,3311419278,1
14,3311992414,3311992419,1
13,3312225025,3311225025,1
12,3312172635,3312172633,1
11,3312150282,3312156282,1
10,3311684368,3311684968,1
9,3310023176,3310023177,1
15,3312285639,3312283639,1
8,3311857943,3311857443,1
6,3311360177,3311360173,1


In [99]:
def data_quality_report(df):
    total = len(df)

    # Cùng UID nhưng nhiều Phone
    uid_phone = (
        df.dropna(subset=["UID_clean", "Phone_clean"])
          .groupby("UID_clean")["Phone_clean"]
          .nunique()
    )
    same_uid_diff_phone = df["UID_clean"].isin(uid_phone[uid_phone > 1].index).sum()

    # Cùng Phone nhưng nhiều UID
    phone_uid = (
        df.dropna(subset=["UID_clean", "Phone_clean"])
          .groupby("Phone_clean")["UID_clean"]
          .nunique()
    )
    same_phone_diff_uid = df["Phone_clean"].isin(phone_uid[phone_uid > 1].index).sum()

    # Thiếu UID hoặc Phone
    missing_uid = df["UID_clean"].isna().sum()
    missing_phone = df["Phone_clean"].isna().sum()

    print(f"Total rows: {total}")
    print(f"Same UID, different Phone : {same_uid_diff_phone} ({same_uid_diff_phone/total:.2%})")
    print(f"Same Phone, different UID : {same_phone_diff_uid} ({same_phone_diff_uid/total:.2%})")
    print(f"Missing UID               : {missing_uid} ({missing_uid/total:.2%})")
    print(f"Missing Phone             : {missing_phone} ({missing_phone/total:.2%})")

In [100]:
print("=== LEADS ===")
data_quality_report(df_leads)

print("\n=== GMV ===")
data_quality_report(df_gmv)

=== LEADS ===
Total rows: 16336
Same UID, different Phone : 827 (5.06%)
Same Phone, different UID : 551 (3.37%)
Missing UID               : 0 (0.00%)
Missing Phone             : 0 (0.00%)

=== GMV ===
Total rows: 463
Same UID, different Phone : 6 (1.30%)
Same Phone, different UID : 0 (0.00%)
Missing UID               : 0 (0.00%)
Missing Phone             : 0 (0.00%)


In [101]:
print(df_leads["UID_clean"].dtype)
print(df_gmv["UID_clean"].dtype)

object
object


In [107]:
df_leads["UID_clean"] = df_leads["UID_clean"].astype(str)
df_gmv["UID_clean"] = df_gmv["UID_clean"].astype(str)

In [108]:
uid_check = (
    df_leads[["UID_clean", "Phone_clean"]]
    .merge(
        df_gmv[["UID_clean", "Phone_clean"]],
        on="UID_clean",
        suffixes=("_lead", "_gmv")
    )
)

uid_diff_phone = uid_check[
    uid_check["Phone_clean_lead"] != uid_check["Phone_clean_gmv"]
]

print(len(uid_diff_phone))
uid_diff_phone.head()

0


,UID_clean,Phone_clean_lead,Phone_clean_gmv


In [105]:
df_leads["Phone_clean"] = df_leads["Phone_clean"].astype(str)
df_gmv["Phone_clean"] = df_gmv["Phone_clean"].astype(str)

In [106]:
phone_check = (
    df_leads[["UID_clean", "Phone_clean"]]
    .merge(
        df_gmv[["UID_clean", "Phone_clean"]],
        on="Phone_clean",
        suffixes=("_lead", "_gmv")
    )
)

phone_diff_uid = phone_check[
    phone_check["UID_clean_lead"] != phone_check["UID_clean_gmv"]
]

print(len(phone_diff_uid))
phone_diff_uid.head()

26


,UID_clean_lead,Phone_clean,UID_clean_gmv
14,,817090779689,3300978231
43,3311193217,817090293668,3311228095
47,,817084801084,3311228463
53,,420773055288,3286214118
107,3311627622,84357978861,3311634513


In [109]:
df_leads.columns

Index(['Đầu Ngày', 'Họ và tên', 'Người trực', 'Số điện thoại', 'Tuổi',
       'Quốc Gia', 'Note', 'TVTS', 'Ad_id', 'Nguồn', 'Quốc gia \nQuảng cáo',
       'Mẫu Quảng cáo', 'Mã CRM', 'Sale sau phân', 'Link FB', 'UID', 'TT',
       'Trạng thái', 'Ngày lên học thử', 'Current Binded Sale',
       'Sales in latest system-weighted allocation',
       'Latest system-weighted Allocation Time', 'Allocate reason',
       'Tên sale chatpage', 'KOL', 'Check Ad_ID', 'UID_clean', 'Phone_clean',
       'Phone_extracted_from_note'],
      dtype='object')

In [ ]:
df_leads = pd.read_csv("./test/")

In [112]:
df_leads["Mã CRM"].astype("string").str.len().value_counts().sort_index()

Mã CRM
3    6544
8    9779
Name: count, dtype: Int64

In [ ]:
df_test_ov_11_cau = pd.read_csv("./data_test/PalFish - Chatpage 2026 - Test OV-11câu 210426.csv", encoding="utf-8")
df_if_ov_lan_2 = pd.read_csv("./data_test/PalFish - Chatpage 2026 - IF OV  (LẦN2).csv", encoding='utf-8')
df_data_gg = pd.read_csv("./data_test/PalFish - Chatpage 2026 - Data GG.csv", encoding='utf-8')
df_data_gg_gop_ov = pd.read_csv("./data_test/PalFish - Chatpage 2026 - Data GG Gộp OV.csv", encoding='utf-8')
df_ls_if_nt = pd.read_csv("./data_test/PalFish - Chatpage 2026 - LS-IF (VN-Nền tảng).csv", encoding='utf-8')
df_ls_if_ntov = pd.read_csv("./data_test/PalFish - Chatpage 2026 -  LS-IF (VN-Nền tảng OV).csv", encoding='utf-8')



In [128]:
col = "Số điện thoại"

samples = (
    df_data_gg_gop_ov.assign(length=df_data_gg_gop_ov[col].astype("string").str.len())
    .groupby("length")[col]
    .apply(lambda x: x.drop_duplicates().head(5).tolist())
)

print(samples)

length
1                                                   [-]
7                                             [5415835]
9     [721888446, 366196187, 969425182, 981655339, 8...
10    [1034881395, 7091505191, 1049250086, 803410280...
11                           [15901976562, 17644508424]
12           [420702813541, 821065608084, 082 244 4818]
Name: Số điện thoại, dtype: object


### UID+Phone, Phone, UID:        281
### Phone_extracted_from_note      3
### Unmatched: 179 có cách nào để liên kết trực tiếp gmv với ads hay không? ví dụ thông qua cổng thanh toán, phương thức thanh toán,...